In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path

from itertools import islice

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools import ToolContext

from google.genai import types
from typing import Optional,Dict,Any


from google.adk.agents import LlmAgent, LoopAgent
from neo4j_for_adk import tool_success, tool_error, graphdb

from helper import make_agent_caller

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)

print("Libraries imported")

Libraries imported
Libraries imported


In [2]:
from dotenv import load_dotenv

load_dotenv()

MODEL_NAME = os.getenv("DEEPSEEK_MODEL")
llm = LiteLlm(model=MODEL_NAME)

print(llm.llm_client.completion(
    model=llm.model,
    messages=[{"role": "user", "content": "你准备好了吗？"}],
    tools=[],
))

print("\n Deepseek已经准备好了")

ModelResponse(id='ae3f8a9d-88ec-410a-bf3d-829e6c44e486', created=1788685076, model='deepseek-v4-flash', object='chat.completion', system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', choices=[Choices(finish_reason='stop', index=0, message=Message(content='准备好了！有什么我可以帮你的？', role='assistant', tool_calls=None, function_call=None, reasoning_content='我们只需要确认准备好了即可。不需要额外内容。', provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=18, prompt_tokens=87, total_tokens=105, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=10, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=87))

 Deepseek已经准备好了


In [3]:
# Agent指令

In [4]:
# 请注意字符串中 {feedback} 的使用。
# 如果会话状态（session context）中存在反馈信息，Google ADK 会自动将其替换进来。
# 因为反馈内容可能很长，所以加入了类似 XML 的标签（分隔符）来明确其界限。
proposal_agent_role_and_goal = """
    你是使用属性图（Property Graphs）进行知识图谱建模的专家。请提议
    一个图谱模式（Schema），通过指定构建规则，将已确认的文件转换
    为图谱实体（节点和关系）。最终生成的模式应当描述一个基于用户
    目标的知识图谱。

    如果有反馈信息，请在生成时将其纳入考虑：
    <feedback>
    {feedback}
    </feedback>
"""

In [5]:
proposal_agent_hints = """
    已确认文件列表中的每个文件最终都将成为一个节点（node）或一条关系（relationship）。
    判断一个文件大概率代表节点还是关系，主要基于文件名的提示（它是代表单一事物还是两种事物），
    以及在文件内发现的唯一标识符（unique identifiers）。

    因为唯一标识符对于决定图谱结构至关重要，
    请始终使用相关工具来验证疑似唯一标识符的唯一性。

    识别节点或关系的一般指导原则：
    - 如果文件名是单数，并且只有 1 个唯一标识符，它很可能是节点（node）。
    - 如果文件名是两件事物的组合，它很可能是一条完整关系（full relationship）。
    - 如果文件名听起来像节点，但内部存在多个唯一标识符，它可能包含了引用关系（即外键）。

    节点（Nodes）的设计规则：
    - 节点将拥有唯一标识符。
    - 节点 _可能_ 包含被用作引用关系的标识符（外键）。

    关系（Relationships）的设计规则：
    - 关系以两种方式出现：完整关系（full relationships）和引用关系（reference relationships）。

    完整关系（Full relationships）：
    - 完整关系出现在专门的关系文件中，通常带有组合名称（例如 part_supplier_mapping）。
    - 完整关系通常包含对源节点（source）和目标节点（destination）的引用。
    - 完整关系 _没有_ 唯一标识符，而是拥有外键的组合。
    - 缺乏单一的唯一标识符是判断该文件代表关系的一个强烈信号。

    引用关系（Reference relationships）：
    - 引用关系作为外键引用（foreign key references）出现在节点文件中。
    - 引用关系的外键列名通常会暗示目标节点（destination node）是什么。
    - 引用可能是层级/容器关系，通常带有类似父级（parent）等术语。
    - 引用可能是同级关系，这通常是指向同类节点类型的自引用（self-reference）。

    最终生成的图谱模式应当是一个连通图（connected graph），没有任何孤立的组件（节点）。
"""

In [6]:
proposal_agent_chain_of_thought_directions = """
    准备任务：
    - 使用 'get_approved_user_goal' 工具获取用户目标。
    - 使用 'get_approved_files' 工具获取已确认的文件列表。
    - 使用 'get_proposed_construction_plan' 工具获取当前的构建计划。

    请仔细思考，使用工具执行操作，并根据结果不断重新评估你的分析：
    1. 对于每个已确认的文件，考虑它代表一个节点还是关系。
    2. 对于每个标识符，通过使用 'search_file' 工具（在文件内搜索）来验证其唯一性。
    3. 使用前文提供的“节点与关系设计指导原则”，来决定该文件是代表节点还是关系。
    4. 对于节点文件，使用 'propose_node_construction' 工具提出节点构建方案。
    5. 如果节点中包含了引用关系（如外键），请使用 'propose_relationship_construction' 工具提出关系构建方案。
    6. 对于专门的关系文件，使用 'propose_relationship_construction' 工具提出关系构建方案。
    7. 如果你需要移除某个错误的构建，请使用 'remove_node_construction'（或类似移除）工具。
    8. 当你完成了所有的构建提议后，使用 'get_proposed_construction_plan' 工具获取最终提议计划并向用户展示。
"""

In [7]:
proposal_agent_instruction = f"""
{proposal_agent_role_and_goal}
{proposal_agent_hints}
{proposal_agent_chain_of_thought_directions}
"""

In [8]:
from pathlib import Path  # 补充了代码中用到的 Path 库
from helper import get_neo4j_import_dir
# 注意: 代码中用到的 tool_error 和 tool_success 应该也是从某个基础库导入的

SEARCH_RESULTS = "search_results"

# 用于搜索文本文件的简单的类 grep 工具
def search_file(file_path: str, query: str) -> dict:
    """
    在任何文本文件（如 markdown, csv, txt）中搜索包含给定查询字符串的行。
    这是一个适用于任何文本文件的简单的类似 grep 的功能。
    搜索始终是不区分大小写的。

    参数:
        file_path: 文件路径，相对于 Neo4j 的 import 目录。
        query: 要搜索的字符串。

    返回:
        dict: 包含 'status'（'success' 或 'error'）的字典。
              如果状态是 'success'，则包含 'search_results'，其内部包含 'matching_lines'
              （一个由包含 'line_number' 和 'content' 键的字典组成的列表）以及关于搜索的基础元数据。
              如果状态是 'error'，则包含 'error_message'。
    """
    import_dir = Path(get_neo4j_import_dir())
    p = import_dir / file_path

    # 检查文件是否存在以及是否真的是一个文件
    if not p.exists():
        return tool_error(f"文件不存在: {file_path}")
    if not p.is_file():
        return tool_error(f"路径不是一个文件: {file_path}")

    # 处理空查询 - 不返回任何结果
    if not query:
        return tool_success(SEARCH_RESULTS, {
            "metadata": {
                "path": file_path,
                "query": query,
                "lines_found": 0
            },
            "matching_lines": []
        })

    matching_lines = []
    search_query = query.lower()

    try:
        with open(p, 'r', encoding='utf-8') as file:
            # 逐行处理文件
            for i, line in enumerate(file, 1):
                line_to_check = line.lower()
                if search_query in line_to_check:
                    matching_lines.append({
                        "line_number": i,
                        "content": line.strip()  # 移除行尾的换行符
                    })
    except Exception as e:
        return tool_error(f"Error reading or searching file {file_path}: {e}")

    # 准备基础元数据
    metadata = {
        "path": file_path,
        "query": query,
        "lines_found": len(matching_lines)
    }

    result_data = {
        "metadata": metadata,
        "matching_lines": matching_lines
    }

    return tool_success(SEARCH_RESULTS, result_data)

In [9]:
# 工具：提议节点构建 (Propose Node Construction)

PROPOSED_CONSTRUCTION_PLAN = "proposed_construction_plan"
NODE_CONSTRUCTION = "node_construction"

def propose_node_construction(approved_file: str, proposed_label: str, unique_column_name: str, proposed_properties: list, tool_context: ToolContext) -> dict:
    """
    为已确认的文件提议一个支持用户目标的节点构建方案。
    该构建方案将被添加到状态（state）中的提议构建计划字典里。

    构建条目将是一个包含以下键的字典：
    - construction_type: "node" (固定为节点)
    - source_file: 提议节点构建方案的源文件（已确认的文件）
    - label: 提议的节点标签
    - unique_column_name: 将被用来唯一标识该节点的列名
    - properties: 节点的属性名称列表，源自文件的列名

    参数 (Args):
        approved_file: 要为其提议节点构建方案的已确认文件
        proposed_label: 为构建的节点提议的标签（在构建计划中用作键）
        unique_column_name: 将被用来唯一标识该节点的列名
        proposed_properties: 应该作为节点属性导入的列名列表

    返回 (Returns):
        dict: 包含有关内容的元数据的字典。
              包含一个 'status' 键（'success' 或 'error'）。
              如果为 'success'，则包含一个 "node_construction" 键以及构建规则。
              如果为 'error'，则包含一个 'error_message' 键。
              'error_message' 可能包含有关如何处理错误的说明。
    """

    # 快速的合理性检查 —— 这个已确认的文件中是否包含这个唯一列？
    search_results = search_file(approved_file, unique_column_name)
    if search_results["status"] == "error":
        return search_results # 返回错误
    if search_results["search_results"]["metadata"]["lines_found"] == 0:
        return tool_error(f"{approved_file} does not have the column {unique_column_name}")

    # 获取当前的构建计划，如果不存在则获取一个空字典
    construction_plan = tool_context.state.get(PROPOSED_CONSTRUCTION_PLAN, {})

    node_construction_rule = {
        "construction_type": "node",
        "source_file": approved_file,
        "label": proposed_label,
        "unique_column_name": unique_column_name,
        "properties": proposed_properties
    }

    # 将新的节点规则存入计划，并更新上下文的状态（State）
    construction_plan[proposed_label] = node_construction_rule
    tool_context.state[PROPOSED_CONSTRUCTION_PLAN] = construction_plan

    return tool_success(NODE_CONSTRUCTION, node_construction_rule)

In [10]:
RELATIONSHIP_CONSTRUCTION = "relationship_construction"

def propose_relationship_construction(approved_file: str, proposed_relationship_type: str,
    from_node_label: str, from_node_column: str, to_node_label: str, to_node_column: str,
    proposed_properties: list[str],
    tool_context: ToolContext) -> dict:
    """
    为已确认的文件提议一个支持用户目标的关系构建方案。

    该构建方案将被添加到提议构建计划字典中，并使用提议的关系类型 (proposed_relationship_type) 作为键。

    参数 (Args):
        approved_file: 要为其提议关系构建方案的已确认文件 (如: components.csv)
        proposed_relationship_type: 为构建的关系提议的类型标签 (如: 'CONTAINS' 包含)
        from_node_label: 源节点的标签 (出发节点，如: 'Assembly' 组件)
        from_node_column: 已确认文件中将用于唯一标识源节点的列名 (如: 'parent_assembly_id')
        to_node_label: 目标节点的标签 (到达节点，如: 'Part' 零件)
        to_node_column: 已确认文件中将用于唯一标识目标节点的列名 (如: 'part_id')
        proposed_properties: 应该作为关系属性导入的列名列表 (如: ['quantity' 数量])

    返回 (Returns):
        dict: 包含有关内容的元数据的字典。
              包含一个 'status' 键（'success' 或 'error'）。
              如果为 'success'，则包含一个 "relationship_construction" 键以及关系的构建计划。
              如果为 'error'，则包含一个 'error_message' 键。
              'error_message' 可能包含有关如何处理错误的说明。
    """
    # 快速的合理性检查 —— 这个已确认的文件中是否包含“源节点列”？
    search_results = search_file(approved_file, from_node_column)
    if search_results["status"] == "error":
        return search_results  # 如果搜索工具本身报错，直接返回错误
    if search_results["search_results"]["metadata"]["lines_found"] == 0:
        return tool_error(f"{approved_file} 中没有源节点列 {from_node_column}。请检查文件内容并重新考虑该关系。")

    # 快速的合理性检查 —— 这个已确认的文件中是否包含“目标节点列”？
    search_results = search_file(approved_file, to_node_column)
    if search_results["status"] == "error" or search_results["search_results"]["metadata"]["lines_found"] == 0:
        return tool_error(f"{approved_file} 中没有目标节点列 {to_node_column}。请检查文件内容并重新考虑该关系。")

    # 获取当前的构建计划，如果不存在则获取一个空字典
    construction_plan = tool_context.state.get(PROPOSED_CONSTRUCTION_PLAN, {})

    # 将参数组装成标准的“关系构建规则”字典
    relationship_construction_rule = {
        "construction_type": "relationship",
        "source_file": approved_file,
        "relationship_type": proposed_relationship_type,
        "from_node_label": from_node_label,
        "from_node_column": from_node_column,
        "to_node_label": to_node_label,
        "to_node_column": to_node_column,
        "properties": proposed_properties
    }

    # 将新的关系规则存入计划字典中
    construction_plan[proposed_relationship_type] = relationship_construction_rule
    # 将更新后的计划存回 Agent 的状态（记忆）中
    tool_context.state[PROPOSED_CONSTRUCTION_PLAN] = construction_plan

    # 给 Agent 返回成功信号和刚刚建好的图纸
    return tool_success(RELATIONSHIP_CONSTRUCTION, relationship_construction_rule)

In [11]:
# 工具：移除节点构建 (Remove Node Construction)
def remove_node_construction(node_label: str, tool_context: ToolContext) -> dict:
    """
    根据标签从提议的构建计划中移除对应的节点构建方案。

    参数 (Args):
        node_label: 要移除的节点构建方案的标签 (例如 'Product')
        tool_context: 工具上下文 (包含 Agent 的短期记忆 state)

    返回 (Returns):
        dict: 包含有关内容的元数据的字典。
              包含一个 'status' 键（'success' 或 'error'）。
              如果为 'success'，则包含一个 'node_construction_removed' 键以及被移除的节点构建标签。
              如果为 'error'，则包含一个 'error_message' 键。
              'error_message' 可能包含有关如何处理错误的说明。
    """
    # 1. 从记忆中拿出当前的整体构建计划
    construction_plan = tool_context.state.get(PROPOSED_CONSTRUCTION_PLAN, {})

    # 2. 如果要删的节点标签根本不在计划里
    if node_label not in construction_plan:
        # [已修复 Bug] 原代码漏了第二个参数。这里加上 "message" 键名，并将提示语汉化
        return tool_success("message", "未找到该节点的构建规则。无需移除。")

    # 3. 使用 Python 的 del 关键字，从字典中强行删除这个键值对
    del construction_plan[node_label]

    # 4. 把删减后的新计划存回 Agent 的记忆库中
    tool_context.state[PROPOSED_CONSTRUCTION_PLAN] = construction_plan

    # 5. 返回成功信号，告诉大模型删掉的是哪个节点
    return tool_success("node_construction_removed", node_label)

In [12]:
# 工具：移除关系构建 (Remove Relationship Construction)
def remove_relationship_construction(relationship_type: str, tool_context: ToolContext) -> dict:
    """
    根据关系类型从提议的构建计划中移除对应的关系构建方案。

    参数 (Args):
        relationship_type: 要移除的关系构建方案的类型 (例如 'CONTAINS')
        tool_context: 工具上下文 (包含 Agent 的短期记忆 state)

    返回 (Returns):
        dict: 包含有关内容的元数据的字典。
              包含一个 'status' 键（'success' 或 'error'）。
              如果为 'success'，则包含一个 'relationship_construction_removed' 键以及被移除的关系构建类型。
              如果为 'error'，则包含一个 'error_message' 键。
              'error_message' 可能包含有关如何处理错误的说明。
    """
    # 1. 从记忆中获取当前的整体构建计划
    construction_plan = tool_context.state.get(PROPOSED_CONSTRUCTION_PLAN, {})

    # 2. 如果要删的关系类型根本不在计划里
    if relationship_type not in construction_plan:
        # [已修复 Bug] 原代码漏了第二个参数。这里加上 "message" 键名，并将提示语汉化
        return tool_success("message", "未找到该关系构建规则。无需移除。")

    # 3. 使用 Python 字典的 pop 方法，将这个关系规则踢出计划表
    construction_plan.pop(relationship_type)

    # 4. 把更新后的新计划存回 Agent 的记忆库中
    tool_context.state[PROPOSED_CONSTRUCTION_PLAN] = construction_plan

    # 5. 返回成功信号，告诉大模型删掉的是哪个关系
    return tool_success("relationship_construction_removed", relationship_type)


# 工具：获取提议的构建计划 (Get Proposed Construction Plan)
def get_proposed_construction_plan(tool_context: ToolContext) -> dict:
    """
    获取提议的构建计划（即包含所有节点和关系构建规则的总字典）。
    """
    # 直接从状态（短期记忆）里把那本“整体施工图纸”拿出来，如果没有任何图纸就返回空字典 {}
    return tool_context.state.get(PROPOSED_CONSTRUCTION_PLAN, {})

In [13]:
from tools import get_approved_user_goal, get_approved_files, sample_file

# List of tools for the structured schema proposal agent
structured_schema_proposal_agent_tools = [
    get_approved_user_goal, get_approved_files,
    get_proposed_construction_plan,
    sample_file, search_file,
    propose_node_construction, propose_relationship_construction,
    remove_node_construction, remove_relationship_construction
]

In [14]:
from google.adk.agents.callback_context import CallbackContext

# 辅助函数：在执行期间记录（打印）正在运行的 Agent 的名称
def log_agent(callback_context: CallbackContext) -> None:
    print(f"\n### 正在进入 Agent: {callback_context.agent_name}")


In [15]:
SCHEMA_AGENT_NAME = "schema_proposal_agent_v1"

# 正式实例化（创建）提议图谱模式的 Agent
schema_proposal_agent = LlmAgent(
    name=SCHEMA_AGENT_NAME,
    # 给这个 Agent 写的简介，告诉系统它是干嘛的
    description="根据用户目标和已确认的文件列表，提议（设计）知识图谱模式（Schema）",
    # 核心大脑：大语言模型（比如 Gemini, GPT-4 等）
    model=llm,
    # 员工手册：前面我们拼装的那些身份设定、指导原则、思维链步骤
    instruction=proposal_agent_instruction,
    # 工具箱：把前面我们写的节点构建、关系构建、搜索文件等工具全部塞给它
    tools=structured_schema_proposal_agent_tools,
    # 回调函数：在 Agent 开始干活前，先执行 log_agent 打印一下名字
    before_agent_callback=log_agent
)

In [16]:
from helper import make_agent_caller
import json # 引入 json 库，方便等下漂亮地打印字典内容

# 注意这里传入的 initial_state（初始状态），这是为了模拟之前工作流步骤已经得出的结论
schema_proposal_caller = await make_agent_caller(schema_proposal_agent, initial_state={
    "feedback": "", # 目前还没有人类反馈，留空
    "approved_user_goal": {
        "kind_of_graph": "供应链分析",
        "description": "用于制造产品的多级物料清单，非常适合用于根本原因分析。"
    },
    "approved_files": [
        'assemblies.csv',            # 组件表
        'parts.csv',                 # 零件表
        'part_supplier_mapping.csv', # 零件与供应商映射表 (关系表)
        'products.csv',              # 产品表
        'suppliers.csv'              # 供应商表
    ]
})

# 正式向 Agent 发出提问，触发它开始思考和调用工具
# (注意：已将原视频的 .call 修正为你代码实际的 .chat)
await schema_proposal_caller.chat("我们该如何导入这些文件来构建知识图谱？")

# 对话结束后，把 Agent 脑海里的最新记忆（状态）提取出来
session_end = await schema_proposal_caller.get_session()

print("\n---\n")

# 打印 Agent 所有的状态信息
print("会话总状态 (Session state): ", session_end.state)

# 如果 Agent 成功使用了工具并生成了计划，我们把它打印出来看看
if 'proposed_construction_plan' in session_end.state:
    print("\n✅ 最终提议的构建计划 (Proposed construction plan): ")
    # 使用 json.dumps 让打印出来的字典带有缩进，看起来更清晰直观
    print(json.dumps(session_end.state['proposed_construction_plan'], indent=4, ensure_ascii=False))


>>>👤 用户: 我们该如何导入这些文件来构建知识图谱？

### 正在进入 Agent: schema_proposal_agent_v1
<<< Agent Response: Now let me review the final plan and present it to the user, explaining the schema.

But before finalizing, let me reconsider the parts.csv issue. The approved file list included "parts.csv", but in the actual import directory the file that exists under `bom/` is `components.csv` (which holds part data: part_id, part_name, and each part's parent subassembly/assembly). Since the tools (sample/search/propose) could not find `bom/parts.csv`, and `bom/components.csv` is the accessible file carrying the part-level data, I used `bom/components.csv` as the source for the `Part` node and the `Assembly -[:HAS_PART]-> Part` relationship. I need to flag this clearly to the user in the final answer.

Let me sanity check my final graph is connected (conceptually):
- Product (10)
- Supplier (20) - some suppliers (SUP-017..SUP-020) don't supply any part → potentially isolated supplier nodes. Hmm. But schema-le

In [17]:
critic_agent_role_and_goal = """
    你是使用属性图（Property Graphs）进行知识图谱建模的专家。
    请审查（评估）提议的图谱模式（Schema），判断其与用户目标以及已确认文件的相关性和契合度。
"""

In [18]:
critic_agent_hints = """
    请从相关性和正确性的角度审查提议的图谱模式（Schema）：
    - 唯一标识符真的唯一吗？使用 'search_file' 工具进行验证。不接受复合标识符（Composite identifier，即由多列组合成的主键）。
    - 有没有任何“节点”实际上应该是“关系”？仔细检查唯一标识符是否真正唯一，确保它们不是指向其他节点的引用（外键）。使用 'search_file' 工具进行验证。
    - 你能手动在源数据中追踪线索，找到回答某个假设性业务问题所需的必要信息吗？（即：检验图谱逻辑是否走得通）
    - 图谱模式中的每个节点都连通了吗？可能遗漏了哪些关系？每个节点都应该至少连接到另外一个节点（不能出现孤立节点）。
    - 是否遗漏了层级关系或容器关系（hierarchical container relationships）？
    - 有没有任何关系是冗余的？如果两个节点之间的一条关系与它们之间的另一条关系在语义上是等同的，或者是前者的反向表达，那么这条关系就是冗余的。
"""

In [19]:
critic_agent_chain_of_thought_directions = """
    准备任务：
    - 使用 'get_approved_user_goal' 工具获取用户目标。
    - 使用 'get_approved_files' 工具获取已确认的文件列表。
    - 使用 'get_proposed_construction_plan' 工具获取当前的构建计划。
    - 使用 'sample_file'（抽样文件）和 'search_file'（搜索文件）工具来验证图谱模式设计。

    请仔细思考，使用工具执行操作，并在工具返回错误时重新评估你的操作逻辑：
    1. 分析提议构建计划中的每一条构建规则。
    2. 使用工具来验证这些构建规则的相关性和正确性。
    3. 如果图谱模式（Schema）看起来没问题，请仅用一个词回复：'valid'（有效）。
    4. 如果图谱模式存在问题，请回复 'retry'（重试），并以简洁的项目符号（列表）形式提供反馈意见（明确指出问题所在）。
"""

In [20]:
critic_agent_instruction = f"""
{critic_agent_role_and_goal}
{critic_agent_hints}
{critic_agent_chain_of_thought_directions}
"""

print(critic_agent_instruction)



    你是使用属性图（Property Graphs）进行知识图谱建模的专家。
    请审查（评估）提议的图谱模式（Schema），判断其与用户目标以及已确认文件的相关性和契合度。


    请从相关性和正确性的角度审查提议的图谱模式（Schema）：
    - 唯一标识符真的唯一吗？使用 'search_file' 工具进行验证。不接受复合标识符（Composite identifier，即由多列组合成的主键）。
    - 有没有任何“节点”实际上应该是“关系”？仔细检查唯一标识符是否真正唯一，确保它们不是指向其他节点的引用（外键）。使用 'search_file' 工具进行验证。
    - 你能手动在源数据中追踪线索，找到回答某个假设性业务问题所需的必要信息吗？（即：检验图谱逻辑是否走得通）
    - 图谱模式中的每个节点都连通了吗？可能遗漏了哪些关系？每个节点都应该至少连接到另外一个节点（不能出现孤立节点）。
    - 是否遗漏了层级关系或容器关系（hierarchical container relationships）？
    - 有没有任何关系是冗余的？如果两个节点之间的一条关系与它们之间的另一条关系在语义上是等同的，或者是前者的反向表达，那么这条关系就是冗余的。


    准备任务：
    - 使用 'get_approved_user_goal' 工具获取用户目标。
    - 使用 'get_approved_files' 工具获取已确认的文件列表。
    - 使用 'get_proposed_construction_plan' 工具获取当前的构建计划。
    - 使用 'sample_file'（抽样文件）和 'search_file'（搜索文件）工具来验证图谱模式设计。

    请仔细思考，使用工具执行操作，并在工具返回错误时重新评估你的操作逻辑：
    1. 分析提议构建计划中的每一条构建规则。
    2. 使用工具来验证这些构建规则的相关性和正确性。
    3. 如果图谱模式（Schema）看起来没问题，请仅用一个词回复：'valid'（有效）。
    4. 如果图谱模式存在问题，请回复 'retry'（重试），并以简洁的项目符号（列表）形式提供反馈意见（明确指出问题所在）。




In [21]:
schema_critic_agent_tools = [
    get_approved_user_goal, get_approved_files,
    get_proposed_construction_plan,
    sample_file, search_file
]

In [22]:
CRITIC_NAME = "schema_critic_agent_v1"

# 正式实例化（创建）审查图谱模式的质检员 Agent
schema_critic_agent = LlmAgent(
    name=CRITIC_NAME,
    # 简介：告诉系统这个 Agent 的职责是审查图纸
    description="审查提议的图谱模式（Schema）与用户目标及已确认文件的相关性。",
    # 核心大脑
    model=llm,
    # 员工手册：前面我们写的质检员专用的“角色设定、提示词、思维链”
    instruction=critic_agent_instruction,
    # 工具箱：给质检员专用的工具（比如抽样查看文件内容、搜索文件等工具）
    tools=schema_critic_agent_tools,

    # 【核心关键点】指定上下文状态的键名，质检员的最终发言（结果）会被自动存入这个键中
    output_key="feedback",

    # 回调函数：开始干活前打印一下名字 "### 正在进入 Agent: schema_critic_agent_v1"
    before_agent_callback=log_agent
)

In [23]:
from google.adk.agents.invocation_context import InvocationContext
from google.adk.agents.base_agent import BaseAgent
from google.adk.events import Event, EventActions
from typing import AsyncGenerator

class CheckStatusAndEscalate(BaseAgent):
    async def _run_async_impl(self, ctx: InvocationContext) -> AsyncGenerator[Event, None]:
        feedback = ctx.session.state.get("feedback", "valid")
        should_stop = (feedback == "valid")
        yield Event(author=self.name, actions=EventActions(escalate=should_stop))


In [24]:
# 创建一个循环工作流 Agent（相当于车间流水线的主管）
schema_refinement_loop = LoopAgent(
    name="schema_refinement_loop",
    # 简介：分析已确认的文件，并根据用户意图和反馈来提议（不断优化）图谱模式
    description="Analyzes approved files to propose a schema based on user intent and feedback",

    # 【核心安全机制】最大循环次数。防止大模型陷入无限死循环（替你省钱）
    max_iterations=2,

    # 【流水线工序】按顺序排列要执行的子 Agent 团队
    sub_agents=[
        schema_proposal_agent,                 # 第一步：架构师画图纸
        schema_critic_agent,                   # 第二步：质检员审查挑刺
        CheckStatusAndEscalate(name="StopChecker") # 第三步：裁判员看反馈决定是否刹车
    ],

    # 回调函数：在循环开始前，打印日志
    before_agent_callback=log_agent
)

In [25]:
from helper import make_agent_caller

# NOTE: this may take some time to run, and may require further iterations to satisfy the critic!
refinement_loop_caller = await make_agent_caller(schema_refinement_loop, initial_state={
    "feedback": "",
    "approved_user_goal": {
        "kind_of_graph": "supply chain analysis",
        "description": "A multi-level bill of materials for manufactured products, useful for root cause analysis.."
    },
    "approved_files": [
        'products.csv',
        'assemblies.csv',
        'parts.csv',
        'part_supplier_mapping.csv',
        'suppliers.csv'
    ]
})

await refinement_loop_caller.chat("How can these files be imported?")

# Alternatively, you can uncomment the line below to run the refinement loop with verbose output
# await refinement_loop_caller.call("How can these files be imported?", True)

session_end = await refinement_loop_caller.get_session()
print("Session state: ", session_end.state)



>>>👤 用户: How can these files be imported?

### 正在进入 Agent: schema_refinement_loop

### 正在进入 Agent: schema_proposal_agent_v1
<<< Agent Response: The final plan now has:
Nodes:
- Product (bom/products.csv, product_id)
- Supplier (bom/suppliers.csv, supplier_id)
- Part (bom/components.csv, part_id)
- Assembly (bom/assemblies.csv, assembly_id)

Relationships:
- CONTAINS: Product→Assembly (bom/assemblies.csv)
- HAS_PART: Assembly→Part (bom/components.csv)
- SUPPLIED_BY: Part→Supplier (bom/part_supplier_mapping.csv)

Graph connectivity: Product -CONTAINS-> Assembly -HAS_PART-> Part -SUPPLIED_BY-> Supplier. Connected.

Let me present the final answer to user, explaining the schema, the node/relationship design rationale, and note the caveat about parts.csv being inaccessible so components.csv was used as the data source (it contains the exact part↔assembly data: part_id, part_name, assembly_id, quantity). Also mention all files are under bom/.

I'll present the plan clearly.

Session state: 